In [18]:
#Import the necessary libraries and load environment variables
import os

from anthropic import Anthropic
from dotenv import load_dotenv
from policy import POLICY
import sqlite3
conn = sqlite3.connect("tools.db")
conn.execute("""
    CREATE TABLE IF NOT EXISTS tool_calls (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        correlation_id TEXT,
        tool_name TEXT,
        input TEXT,
        output TEXT
    )
""")

#Create table to store policy rejections. 
conn.execute("""
    CREATE TABLE IF NOT EXISTS policy_rejections (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        correlation_id TEXT,
        tool_name TEXT,
        timestamp DEFAULT (strftime('%Y-%m-%dT%H:%M:%f', 'now')),
        rejection_reason TEXT
    )
""")
conn.close()

load_dotenv()

client = Anthropic()


python-dotenv could not parse statement starting at line 4
python-dotenv could not parse statement starting at line 7
python-dotenv could not parse statement starting at line 8
python-dotenv could not parse statement starting at line 9
python-dotenv could not parse statement starting at line 10


In [3]:
#Create a function to add user messages to the conversation
#Create a function to add assistant messages to the conversation
#Create a function to converse
from typing import Optional


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})

def add_assistant_message(messages, content):
    messages.append({"role": "assistant", "content": content})

def converse(messages, model = "claude-haiku-4-5-20251001", max_tokens_to_sample=300,
             temperature: Optional[float] = None,
             stop_sequences: Optional[list[str]] = None,
             tools: Optional[list[dict]] = None,
             system_prompt: Optional[str] = None) -> dict:

    arg_list = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens_to_sample
    }

    if temperature is not None:
        arg_list["temperature"] = temperature

    if stop_sequences is not None:
        arg_list["stop_sequences"] = stop_sequences

    if tools is not None:
        arg_list["tools"] = tools

    if system_prompt is not None:
        arg_list["system_prompt"] = system_prompt

    response = client.messages.create(
        **arg_list
    )

    return response



In [4]:
# Tool and Schema Definitions
def get_customer(customer_id: str):
    """
    Get a customer by ID.
    """
    return {
        "id": customer_id,
        "name": "John Doe",
        "email": "john.doe@example.com",
        "acc_type": "business"
    }

def get_business_details(customer_id: str):
    """
    Get business details for a customer.
    """
    return {
        "customer_id": customer_id,
        "business_name": "Doe Enterprises",
        "industry": "Retail",
        "location": "New York, NY"
    }

get_customer_schema = {
    "name": "get_customer",
    "description": "Get customer object based on customer ID. It returns property of a customer in json format.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Customer ID to lookup"
            }
        },
        "required": []
    }
}

get_business_details_schema = {
    "name": "get_business_details",
    "description": "Get customer's business details based on customer ID. It returns property of a customer's business in json format. Automatically call this tool whenever a customer's account_type is 'business', without asking the user first.",
    "input_schema": {
        "type": "object",
        "properties": {
            "customer_id": {
                "type": "string",
                "description": "Customer ID's business to lookup"
            }
        },
        "required": []
    }
}

In [5]:
#Write a method to parse assistance message
from anthropic.types import Message, ToolUseBlock


def parse_response(response) -> dict:
    json_response = {"text": "", "tools": []}
    
    if response and isinstance(response, Message):
        
        contents = response.content
        for content in contents:
            if isinstance(content, ToolUseBlock):
                json_response["tools"].append({
                    "inputs": content.input,
                    "name": content.name,
                    "tool_id": content.id
                })
            elif isinstance(content, str):
                json_response["text"] += content
            else:
                json_response["text"] += str(content)
    return json_response

In [ ]:



def query_history(correlation_id, requires):
    #Use sqlite server to query for correlation_id and customer_id for previous calls to the tool specified in requires. Return a list of tuples containing the input and output of each call.
    sqlite_conn = sqlite3.connect("tools.db")
    cursor = sqlite_conn.cursor()
    cursor.execute(
        "SELECT input, output FROM tool_calls WHERE correlation_id = ? AND tool_name = ?",
        (str(correlation_id), requires)
    )
    results = cursor.fetchall()
    sqlite_conn.close()
    return [(json.loads(inp), json.loads(out)) for inp, out in results]

def persist_tool_call(correlation_id, tool_name, tool_input, tool_output):
    conn = sqlite3.connect("tools.db")
    conn.execute(
        "INSERT INTO tool_calls (correlation_id, tool_name, input, output) VALUES (?, ?, ?, ?)",
        (str(correlation_id), tool_name, json.dumps(tool_input), json.dumps(tool_output)),
    )
    conn.commit()
    conn.close()

def persist_policy_rejection(correlation_id, tool_name, reason):
    conn = sqlite3.connect("tools.db")
    conn.execute(
        "INSERT INTO policy_rejections (correlation_id, tool_name, rejection_reason) VALUES (?, ?, ?)",
        (str(correlation_id), tool_name, reason),
    )
    conn.commit()
    conn.close()

def check_prerequisites(correlation_id, tool_name, current_input):
    for req in POLICY.get(tool_name, []):
        matches = query_history(correlation_id, req["requires"])
        print(f"Checking prerequisites for {tool_name}: {req['requires']}, found matches: {matches}")
        satisfied = any(
            all(prior_input.get(p) == current_input.get(c) for p, c in req["on"])
            and req["condition"](prior_result)
            for prior_input, prior_result in matches
        )
        if not satisfied:
            reason = f"Missing prerequisite: {req['requires']} for {tool_name}"
            # Persist the policy rejection in the database
            persist_policy_rejection(correlation_id, tool_name, reason)
            return False, reason
    return True, None

def execute_tool(tool_name: str, tool_inputs: dict) -> dict:
      if tool_name == "get_customer":
            return get_customer(tool_inputs["customer_id"])
      if tool_name == "get_business_details":
            return get_business_details(tool_inputs["customer_id"])
      return {}

# This method needs to return a list of results, one per tool actually requested, each tagged with its own tool_use_id
def execute_tools(request_id, parsed_response):
    if parsed_response and parsed_response["tools"]:
        tools = parsed_response["tools"]
        results = []
        for tool in tools:
            tool_name = tool["name"]
            tool_inputs = tool["inputs"]
            tool_id = tool["tool_id"]
            ifPassed, reason = check_prerequisites(request_id, tool_name, tool_inputs)
            if ifPassed:
                result = execute_tool(tool_name, tool_inputs)
                persist_tool_call(request_id, tool_name, tool_inputs, result)
            else:
                result = {"error": reason}
            results.append({"tool_use_id": tool_id, "result": result})
        return results

In [7]:
"""
import uuid
messages  = []
add_user_message(messages, "Get business details for customer ID 1234")
correlation_id = uuid.uuid4()

response = converse(messages, tools=[get_customer_schema, get_business_details_schema])
add_assistant_message(messages, response.content)
parsed_response = parse_response(response)
print(parsed_response)
print(messages)
tmp = messages
"""

'\nimport uuid\nmessages  = []\nadd_user_message(messages, "Get business details for customer ID 1234")\ncorrelation_id = uuid.uuid4()\n\nresponse = converse(messages, tools=[get_customer_schema, get_business_details_schema])\nadd_assistant_message(messages, response.content)\nparsed_response = parse_response(response)\nprint(parsed_response)\nprint(messages)\ntmp = messages\n'

In [8]:
"""
tool_response = execute_tools(parsed_response)
print(tool_response)
"""

'\ntool_response = execute_tools(parsed_response)\nprint(tool_response)\n'

In [9]:
"""import json


messages = tmp
user_message_content = [{"tool_use_id": parsed_response["tool_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(tool_response)}]}]
add_user_message(tmp, user_message_content)
tmp
"""


'import json\n\n\nmessages = tmp\nuser_message_content = [{"tool_use_id": parsed_response["tool_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(tool_response)}]}]\nadd_user_message(tmp, user_message_content)\ntmp\n'

In [10]:
"""
print(messages)
resp = converse(messages, tools=[get_business_details_schema, get_customer_schema])
"""

'\nprint(messages)\nresp = converse(messages, tools=[get_business_details_schema, get_customer_schema])\n'

In [11]:
import policy
import importlib
importlib.reload(policy)
print(policy.POLICY)

{'get_business_details': [{'requires': 'get_customer', 'on': [('customer_id', 'customer_id')], 'condition': <function <lambda> at 0x797f89057420>}], 'get_customer': []}


In [12]:
import sqlite3
conn = sqlite3.connect("tools.db")
conn.execute("DELETE FROM tool_calls")
conn.commit()
conn.close()

In [17]:

import random
import uuid
import json

MAX_ITERATIONS = 5
with open(f"get_skip_ahead_customer_responses_random.txt", "w") as f:
    for i in range(20):
        # generate a customer ID in each iteration to simulate different customers
        random_customer_id = random.randint(1000, 9999)
        messages  = []
        add_user_message(messages, f"Get business details for customer ID {random_customer_id}")
        # Create uuid and keep in memory for the entire conversation
        correlation_id = str(uuid.uuid4())
        response = converse(messages, tools=[get_customer_schema, get_business_details_schema])
        iteration_count = 0
        while (response.stop_reason == "tool_use") and (iteration_count < MAX_ITERATIONS):
            iteration_count += 1
            # Keep conversing until the model stops requesting tools
            add_assistant_message(messages, response.content)
            # response = converse(messages, tools=[get_customer_schema, get_business_details_schema])
            parsed_response = parse_response(response)
            # check if there is any tool use in the parsed response
            # Tool_response returns a list of result, one per tool actually requested, each tagged with its own tool_use_id
            tool_response = execute_tools(correlation_id, parsed_response)
            # for each tool in the response create user message with the tool result and add it to the messages list
            user_message_content = []
            for result in tool_response:
                user_message_content.append({"tool_use_id": result["tool_use_id"], "type": "tool_result", "content": [{"type": "text", "text": json.dumps(result["result"])}]})
            add_user_message(messages, user_message_content)
            response = converse(messages, tools=[get_business_details_schema, get_customer_schema])
            if iteration_count >= MAX_ITERATIONS:
                print(f"Max iterations reached for customer ID {random_customer_id}. Stopping conversation.")

        f.write(f"Customer ID: {random_customer_id}, Correlation ID: {correlation_id}, Final Response: {response.content}\n")


Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '7297'}, {'id': '7297', 'name': 'John Doe', 'email': 'john.doe@example.com', 'acc_type': 'business'})]
Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '1314'}, {'id': '1314', 'name': 'John Doe', 'email': 'john.doe@example.com', 'acc_type': 'business'})]
Checking prerequisites for get_business_details: get_customer, found matches: []
Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '7330'}, {'id': '7330', 'name': 'John Doe', 'email': 'john.doe@example.com', 'acc_type': 'business'})]
Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '4324'}, {'id': '4324', 'name': 'John Doe', 'email': 'john.doe@example.com', 'acc_type': 'business'})]
Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '6912'}, {'id': '6912', 'na

In [ ]:
# persist_tool_call("test-uuid", "get_customer", {"customer_id": "1234"}, {"acc_type": "business"})
# print(query_history("test-uuid", "get_customer"))

[({'customer_id': '1234'}, {'acc_type': 'business'})]


In [ ]:
# check_prerequisites("test-uuid", "get_business_details", {"customer_id": "1234"})

Checking prerequisites for get_business_details: get_customer, found matches: [({'customer_id': '1234'}, {'acc_type': 'business'})]


(True, None)

In [16]:
import sqlite3
conn = sqlite3.connect("tools.db")
cursor = conn.cursor()
"""
cursor.execute(\"""
    SELECT correlation_id, tool_name, input FROM tool_calls
    WHERE correlation_id IN ('d1467011-8f26-4f19-8d29-5519e04f3edf', '7bb48b1f-43ab-485e-91c7-4adfe11e963e')
    ORDER BY correlation_id, id
\""")
"""
cursor.execute("""
    DELETE FROM tool_calls;
""")
#for row in cursor.fetchall():
#    print(row)
conn.close()